# RQ3-03 — Retrain ModernBERT & DeBERTa on Leakage-Free Data

Bám notebook nền `06_modernBERT_deBERTa.ipynb` về **model + hyperparameter chính**:

- ModernBERT: `answerdotai/ModernBERT-base`
- DeBERTa: `microsoft/deberta-v3-base`
- `MAX_LEN=256`, batch size `16`, LR `2e-5`, weight decay `0.01`, tối đa `5` epochs.
- DeBERTa giữ fix nền: **float32**, `fp16=False`, `bf16=False`, `tf32=False`, `max_grad_norm=1.0`.

### Sửa protocol để RQ3 không leakage
Notebook 06 cũ dùng 20% Test làm `eval_dataset` cho Early Stopping. RQ3 dùng protocol 2-stage đã được nhóm chuẩn hóa ở notebook 07:

1. Outer split: 80% Train Pool / 20% internal Test.
2. Stage A: 90% inner-train / 10% validation trong Train Pool → chọn best epoch.
3. Stage B: khởi tạo lại pretrained model, retrain trên **full 80% Train Pool** đúng số epoch đã chọn.
4. Internal Test chỉ dùng một lần ở cuối.
5. Save final model để notebook external RQ3 chỉ inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Nếu thiếu package:
# %pip install -U pandas numpy scikit-learn torch transformers accelerate

import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

SEED = 42
TEST_SIZE = 0.20
INNER_VAL_SIZE = 0.10
MAX_LEN = 256
BATCH_SIZE = 16
MAX_EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
if Path("/content/AI-Generated-Text-Essay-Detection").exists():
    PROJECT_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")

candidates = [
    PROJECT_ROOT / "data" / "processed" / "data_with_features_minus_1000_human.csv",
    PROJECT_ROOT / "data_with_features_minus_1000_human.csv",
    cwd / "data_with_features_minus_1000_human.csv",
]
DATA_FILE = next((p for p in candidates if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError("Không tìm thấy data_with_features_minus_1000_human.csv")

RESULT_DIR = Path('/content/drive/MyDrive/tmp/results/rq3_retrain_modernbert_deberta')
RESULT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_FILE)[["text", "label"]].dropna().reset_index(drop=True)
df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(int)
df["_row_id"] = np.arange(len(df))

train_pool_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=TEST_SIZE, random_state=SEED, stratify=df["label"]
)
train_pool_df = df.iloc[train_pool_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

inner_idx, val_idx = train_test_split(
    np.arange(len(train_pool_df)), test_size=INNER_VAL_SIZE,
    random_state=SEED, stratify=train_pool_df["label"]
)
inner_train_df = train_pool_df.iloc[inner_idx].reset_index(drop=True)
val_df = train_pool_df.iloc[val_idx].reset_index(drop=True)

assert set(train_pool_df["_row_id"]).isdisjoint(set(test_df["_row_id"]))
assert set(inner_train_df["_row_id"]).isdisjoint(set(val_df["_row_id"]))

for name, part in [("TrainPool",train_pool_df),("InnerTrain",inner_train_df),("Val",val_df),("Test",test_df)]:
    print(name, len(part), part["label"].value_counts().to_dict())

pd.concat([
    train_pool_df[["_row_id","label"]].assign(split="train_80"),
    test_df[["_row_id","label"]].assign(split="test_20"),
]).to_csv(RESULT_DIR / "internal_split_manifest.csv", index=False)

TrainPool 35013 {0: 21017, 1: 13996}
InnerTrain 31511 {0: 18915, 1: 12596}
Val 3502 {0: 2102, 1: 1400}
Test 8754 {0: 5255, 1: 3499}


In [ ]:
class EssayDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len, return_token_type_ids=None):
        self.texts = frame["text"].tolist()
        self.labels = frame["label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.return_token_type_ids = return_token_type_ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            **({} if self.return_token_type_ids is None else {"return_token_type_ids": self.return_token_type_ids}),
        )
        item = {k: torch.tensor(v) for k,v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def metrics_from_logits(labels, logits):
    logits = np.nan_to_num(logits, nan=0.0, posinf=1e4, neginf=-1e4)
    preds = np.argmax(logits, axis=1)
    x = logits - logits.max(axis=1, keepdims=True)
    probs = np.exp(x) / np.exp(x).sum(axis=1, keepdims=True)
    return {
        "Accuracy": accuracy_score(labels, preds),
        "Precision": precision_score(labels, preds, zero_division=0),
        "Recall": recall_score(labels, preds, zero_division=0),
        "F1": f1_score(labels, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(labels, probs[:,1]),
    }, preds, probs[:,1]


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    m, _, _ = metrics_from_logits(labels, logits)
    return {k.lower().replace("-", "_"): v for k,v in m.items()}

In [ ]:
def run_two_stage(model_name, short_name, force_float32=False, no_token_type_ids=False):
    print("\n" + "="*90)
    print(f"{short_name} - CHECKING CHECKPOINTS & INITIALIZING")
    print("="*90)

    # 1. Checkpoint kiểm tra xem model cuối cùng và kết quả đã tồn tại chưa để tránh chạy lại từ đầu
    final_pred_path = RESULT_DIR / f"{short_name.lower()}_internal_test_predictions.csv"
    save_dir = RESULT_DIR / f"{short_name.lower()}_final_model"
    stage_a_history_path = RESULT_DIR / f"{short_name.lower()}_stage_a_history.csv"

    if final_pred_path.exists() and save_dir.exists() and stage_a_history_path.exists():
        print(f"[CHECKPOINT] Đã tìm thấy kết quả đã hoàn thành trước đó của {short_name}!")
        print(f"-> Đang khôi phục kết quả từ: {final_pred_path}")

        # Khôi phục thông tin từ các files đã lưu
        pred_df = pd.read_csv(final_pred_path)
        history_df = pd.read_csv(stage_a_history_path)

        eval_rows = [x for x in history_df.to_dict(orient='records') if "eval_f1" in x and not pd.isna(x["eval_f1"])]
        best_row = max(eval_rows, key=lambda x: x["eval_f1"]) if eval_rows else {"epoch": 1, "eval_f1": 0.0}
        best_epoch = max(1, int(round(float(best_row["epoch"]))))
        best_val_f1 = float(best_row["eval_f1"])

        # Tính toán lại metrics từ các predictions đã có sẵn để trả về
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
        labels = pred_df["label"].values
        preds = pred_df["prediction"].values
        probs = pred_df["prob_ai"].values

        result = {
            "Model": short_name,
            "Pretrained": model_name,
            "Accuracy": accuracy_score(labels, preds),
            "Precision": precision_score(labels, preds, zero_division=0),
            "Recall": recall_score(labels, preds, zero_division=0),
            "F1": f1_score(labels, preds, zero_division=0),
            "ROC-AUC": roc_auc_score(labels, probs),
            "Best_val_F1_stage_A": best_val_f1,
            "Selected_epoch": best_epoch,
            "Max_length": MAX_LEN,
            "Batch_size": BATCH_SIZE,
            "LR": LEARNING_RATE,
        }
        print("[CHECKPOINT SUCCESS] Phục hồi thành công:", result)
        return result

    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    token_type_flag = False if no_token_type_ids else None
    inner_ds = EssayDataset(inner_train_df, tokenizer, MAX_LEN, token_type_flag)
    val_ds = EssayDataset(val_df, tokenizer, MAX_LEN, token_type_flag)

    model_kwargs = {"num_labels": 2, "ignore_mismatched_sizes": True}
    if force_float32:
        model_kwargs["torch_dtype"] = torch.float32
    stage_a_model = AutoModelForSequenceClassification.from_pretrained(model_name, **model_kwargs)

    stage_a_dir = RESULT_DIR / f"{short_name.lower()}_stage_a"
    args_kwargs = dict(
        output_dir=str(stage_a_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )
    if force_float32:
        args_kwargs.update(fp16=False, bf16=False, tf32=False, max_grad_norm=1.0)
    else:
        args_kwargs.update(fp16=torch.cuda.is_available(), bf16=False)

    stage_a_args = TrainingArguments(**args_kwargs)
    stage_a_trainer = Trainer(
        model=stage_a_model,
        args=stage_a_args,
        train_dataset=inner_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )

    print(f"\n--- {short_name} STAGE A: Training with early stopping ---")
    stage_a_trainer.train()

    eval_rows = [x for x in stage_a_trainer.state.log_history if "eval_f1" in x and "epoch" in x]
    if not eval_rows:
        raise RuntimeError("Không lấy được validation F1 history")
    best_row = max(eval_rows, key=lambda x: x["eval_f1"])
    best_epoch = max(1, int(round(float(best_row["epoch"]))))
    best_val_f1 = float(best_row["eval_f1"])
    pd.DataFrame(stage_a_trainer.state.log_history).to_csv(stage_a_history_path, index=False)
    print("Best epoch:", best_epoch, "Best val F1:", best_val_f1)

    del stage_a_trainer, stage_a_model, inner_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"\n--- {short_name} STAGE B: Retraining full 80% with selected {best_epoch} epochs ---")
    set_seed(SEED)
    train_ds = EssayDataset(train_pool_df, tokenizer, MAX_LEN, token_type_flag)
    test_ds = EssayDataset(test_df, tokenizer, MAX_LEN, token_type_flag)
    final_model = AutoModelForSequenceClassification.from_pretrained(model_name, **model_kwargs)

    final_args_kwargs = dict(
        output_dir=str(RESULT_DIR / f"{short_name.lower()}_stage_b"),
        eval_strategy="no",
        save_strategy="no",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=best_epoch,
        weight_decay=WEIGHT_DECAY,
        logging_steps=50,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )
    if force_float32:
        final_args_kwargs.update(fp16=False, bf16=False, tf32=False, max_grad_norm=1.0)
    else:
        final_args_kwargs.update(fp16=torch.cuda.is_available(), bf16=False)

    final_trainer = Trainer(
        model=final_model,
        args=TrainingArguments(**final_args_kwargs),
        train_dataset=train_ds,
    )
    final_trainer.train()
    test_out = final_trainer.predict(test_ds)
    m, pred, prob = metrics_from_logits(test_out.label_ids, test_out.predictions)

    # Lưu trữ model hoàn chỉnh làm checkpoint cuối
    final_trainer.save_model(str(save_dir))
    tokenizer.save_pretrained(str(save_dir))

    pd.DataFrame({
        "_row_id": test_df["_row_id"],
        "label": test_out.label_ids,
        "prediction": pred,
        "prob_ai": prob,
        "text": test_df["text"],
    }).to_csv(final_pred_path, index=False)

    result = {
        "Model": short_name,
        "Pretrained": model_name,
        **m,
        "Best_val_F1_stage_A": best_val_f1,
        "Selected_epoch": best_epoch,
        "Max_length": MAX_LEN,
        "Batch_size": BATCH_SIZE,
        "LR": LEARNING_RATE,
    }
    print(result)

    del final_trainer, final_model, train_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

In [ ]:
modernbert_result = run_two_stage(
    "answerdotai/ModernBERT-base",
    "ModernBERT",
    force_float32=False,
    no_token_type_ids=False,
)


ModernBERT - CHECKING CHECKPOINTS & INITIALIZING
[CHECKPOINT] Đã tìm thấy kết quả đã hoàn thành trước đó của ModernBERT!
-> Đang khôi phục kết quả từ: /content/drive/MyDrive/tmp/results/rq3_retrain_modernbert_deberta/modernbert_internal_test_predictions.csv
[CHECKPOINT SUCCESS] Phục hồi thành công: {'Model': 'ModernBERT', 'Pretrained': 'answerdotai/ModernBERT-base', 'Accuracy': 0.997943797121316, 'Precision': 0.9994261119081779, 'Recall': 0.995427264932838, 'F1': 0.9974226804123711, 'ROC-AUC': np.float64(0.999899495547049), 'Best_val_F1_stage_A': 0.9967776584317938, 'Selected_epoch': 4, 'Max_length': 256, 'Batch_size': 16, 'LR': 2e-05}


In [ ]:
deberta_result = run_two_stage(
    "microsoft/deberta-v3-base",
    "DeBERTa-v3",
    force_float32=True,
    no_token_type_ids=True,
)


DeBERTa-v3 - CHECKING CHECKPOINTS & INITIALIZING


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

Step,Training Loss
50,0.390362
100,0.127642
150,0.041666
200,0.090090
250,0.065387
300,0.057213
350,0.056689
400,0.065508
450,0.057242
500,0.075063


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'Model': 'DeBERTa-v3', 'Pretrained': 'microsoft/deberta-v3-base', 'Accuracy': 0.9926890564313456, 'Precision': 0.9844851904090268, 'Recall': 0.9974278365247213, 'F1': 0.9909142532651902, 'ROC-AUC': np.float64(0.9999218479984359), 'Best_val_F1_stage_A': 0.9971387696709585, 'Selected_epoch': 2, 'Max_length': 256, 'Batch_size': 16, 'LR': 2e-05}


In [ ]:
results = pd.DataFrame([modernbert_result, deberta_result]).sort_values("F1", ascending=False)
path = RESULT_DIR / "modernbert_deberta_internal_results.csv"
results.to_csv(path, index=False)
display(results)
print("Saved:", path)
print("✅ Final models đã được save để RQ3 external notebook chỉ inference.")

,Model,Pretrained,Accuracy,Precision,Recall,F1,ROC-AUC,Best_val_F1_stage_A,Selected_epoch,Max_length,Batch_size,LR
0,ModernBERT,answerdotai/ModernBERT-base,0.997944,0.999426,0.995427,0.997423,0.999899,0.996778,4,256,16,0.00002
1,DeBERTa-v3,microsoft/deberta-v3-base,0.992689,0.984485,0.997428,0.990914,0.999922,0.997139,2,256,16,0.00002


Saved: /content/drive/MyDrive/tmp/results/rq3_retrain_modernbert_deberta/modernbert_deberta_internal_results.csv
✅ Final models đã được save để RQ3 external notebook chỉ inference.


In [ ]:
import shutil

# Đường dẫn tới thư mục bạn muốn nén
folder_to_zip = '/content/drive/MyDrive/tmp/results/rq3_retrain_modernbert_deberta'

# Đường dẫn và tên file zip kết quả (không cần thêm đuôi .zip)
output_filename = '/content/drive/MyDrive/tmp/results/rq3_retrain_modernbert_deberta'


# Thực hiện nén thư mục thành file zip
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"Đã nén thành công thư mục {folder_to_zip} thành file {output_filename}.zip")